In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# 1. Baca data mentah
project_dir = Path.cwd()
if (project_dir / 'data' / 'raw').exists():
    data_dir = project_dir / 'data'
elif (project_dir.parent / 'data' / 'raw').exists():
    data_dir = project_dir.parent / 'data'
else:
    raise FileNotFoundError('Folder data/raw tidak ditemukan dari lokasi notebook saat ini.')

raw_path = data_dir / 'raw' / 'bedtime_screentime_sleep_debt.csv'
df = pd.read_csv(raw_path)

# 2. Hapus Pengenal Langsung (user_id)
df_aman = df.drop(columns=['user_id']).copy()

# 3. Generalisasi Kolom 'age' menjadi 'kelompok_umur'
bins = [0, 24, 34, 44, 54, 100]
labels = ['<25', '25-34', '35-44', '45-54', '55+']
df_aman['kelompok_umur'] = pd.cut(df_aman['age'], bins=bins, labels=labels)
df_aman = df_aman.drop(columns=['age'])

# 4. Fungsi Mengukur Nilai k-Anonymity
def cek_k(data, kolom_quasi):
    grup = data.groupby(kolom_quasi, observed=True).size()
    k_min = grup.min()
    grup_satu = (grup == 1).sum()
    total_grup = len(grup)
    print('--- Uji k-Anonymity ---')
    print(f'Nilai k (kelompok terkecil) : {k_min}')
    print(f'Jumlah kelompok berisi 1 orang: {grup_satu}')
    print(f'Total kelompok kombinasi    : {total_grup}')
    return k_min

# 5. Uji Nilai k pada Kombinasi Quasi-Identifier
quasi_cols = ['gender', 'occupation_type', 'kelompok_umur']
k_hasil = cek_k(df_aman, quasi_cols)

# 6. Simpan Dataset Aman ke data/processed/
output_dir = data_dir / 'processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'sleep_data_processed.csv'
df_aman.to_csv(output_path, index=False)

print(f'\nData bersih berhasil disimpan di {output_path}')
print(f'Target k >= 5 terpenuhi? {k_hasil >= 5}')
print(f'Jumlah baris hasil: {len(df_aman)}')

--- Uji k-Anonymity ---
Nilai k (kelompok terkecil) : 1
Jumlah kelompok berisi 1 orang: 2
Total kelompok kombinasi    : 74

Data bersih berhasil disimpan di c:\Users\h\Documents\Semester 3\Exploratory Data Analys\EDA-GroupX\data\processed\sleep_data_processed.csv
Target k >= 5 terpenuhi? False
Jumlah baris hasil: 8500
